# Análisis de rotación de personal con Python

Segunda parte del proyecto. La primera parte ([`sql/consultas_rotacion.sql`](../sql/consultas_rotacion.sql)) identificó los factores asociados a la renuncia comparando grupos de a dos variables. Este notebook busca:

1. Revisar la relación de **todas** las variables con la renuncia.
2. Separar el efecto propio de cada factor con un **modelo de regresión logística**.
3. Evaluar si el modelo sirve para **anticipar** renuncias.
4. Calcular un **puntaje de riesgo** por empleado y guardarlo en MySQL para el dashboard de Power BI.

**Datos:** dataset *IBM HR Analytics Employee Attrition & Performance* (1.470 empleados, datos ficticios), cargado en MySQL en la parte anterior.

## 0. Librerías

In [ ]:
%pip install pandas sqlalchemy pymysql scikit-learn

## 1. Carga de datos desde MySQL

La contraseña se pide con `getpass` para que no quede escrita en el código.

In [3]:
import pandas as pd
from sqlalchemy import create_engine, URL
from getpass import getpass

url = URL.create(
    "mysql+pymysql",
    username="Francisco",
    password=getpass("Contraseña MySQL: "),
    host="localhost",
    port=3306,
    database="hr_attrition",
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM employees", engine)
df.shape

(1470, 35)

In [4]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

Las 35 columnas tienen 1.470 valores no nulos: no hay datos faltantes.

## 2. Variable objetivo

Se crea `renuncio` (1 = renunció, 0 = sigue). Su promedio es directamente la tasa de renuncia.

In [6]:
df["renuncio"] = (df["Attrition"] == "Yes").astype(int)

In [7]:
round(df["renuncio"].mean() * 100, 1)

np.float64(16.1)

In [8]:
df.groupby("OverTime")["renuncio"].agg(["count", "sum", "mean"])

,count,sum,mean
OverTime,,,
No,1054,110,0.104364
Yes,416,127,0.305288


Los resultados coinciden con la parte en SQL: tasa general de 16,1%, y 30,5% vs 10,4% según horas extra.

## 3. Correlación de cada variable con la renuncia

Se convierte `OverTime` a número (`horas_extra`) y se calcula la correlación de todas las variables numéricas con `renuncio`. Valores positivos indican más renuncia; negativos, menos.

In [9]:
df["horas_extra"] = (df["OverTime"] == "Yes").astype(int)
correlaciones = df.corr(numeric_only=True)["renuncio"].sort_values()
correlaciones

TotalWorkingYears          -0.171063
JobLevel                   -0.169105
YearsInCurrentRole         -0.160545
MonthlyIncome              -0.159840
Age                        -0.159205
YearsWithCurrManager       -0.156199
StockOptionLevel           -0.137145
YearsAtCompany             -0.134392
JobInvolvement             -0.130016
JobSatisfaction            -0.103481
EnvironmentSatisfaction    -0.103369
WorkLifeBalance            -0.063939
TrainingTimesLastYear      -0.059478
DailyRate                  -0.056652
RelationshipSatisfaction   -0.045872
YearsSinceLastPromotion    -0.033019
Education                  -0.031373
PercentSalaryHike          -0.013478
EmployeeNumber             -0.010577
HourlyRate                 -0.006846
PerformanceRating           0.002889
MonthlyRate                 0.015170
NumCompaniesWorked          0.043494
DistanceFromHome            0.077924
horas_extra                 0.246118
renuncio                    1.000000
EmployeeCount                    NaN
S

**Observaciones:**

- **Horas extra** es la variable más relacionada con la renuncia (0,25), muy por sobre las demás.
- Un grupo de variables tiene correlaciones casi iguales (-0,16 a -0,17): años de experiencia, nivel del cargo, años en el cargo, sueldo, edad y años con el mismo jefe. Todas reflejan **qué tan avanzada está la persona en su carrera**, por lo que probablemente están relacionadas entre sí (se revisa en la sección 5).
- Aparecen factores no revisados en SQL: **acciones de la empresa** (`StockOptionLevel`, -0,14) y **involucramiento y satisfacción** (-0,10 a -0,13).
- El **desempeño** (`PerformanceRating`) no tiene relación con la renuncia: se van tanto empleados de alto como de bajo desempeño.
- `EmployeeNumber` es un identificador, por lo que su correlación no tiene significado.
- `EmployeeCount` y `StandardHours` dan `NaN` porque tienen el mismo valor para todos los empleados.

## 4. Limpieza: columnas constantes

Se buscan las columnas con un solo valor distinto, que no aportan información.

In [12]:
df.nunique().sort_values().head(6)

EmployeeCount        1
Over18               1
StandardHours        1
Attrition            2
OverTime             2
PerformanceRating    2
dtype: int64

In [13]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18"])
df.shape

(1470, 34)

Se eliminan `EmployeeCount`, `StandardHours` y `Over18`. `EmployeeNumber` se conserva como identificador para cruzar resultados en Power BI, pero no se usa en el modelo. Estos cambios solo afectan al notebook; la tabla en MySQL queda intacta.

## 5. Variables de carrera: multicolinealidad

Se revisa si las variables de carrera están correlacionadas entre sí.

In [14]:
carrera = [
    "Age",
    "TotalWorkingYears",
    "JobLevel",
    "MonthlyIncome",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsWithCurrManager",
]

df[carrera].corr().round(2)

,Age,TotalWorkingYears,JobLevel,MonthlyIncome,YearsAtCompany,YearsInCurrentRole,YearsWithCurrManager
Age,1.00,0.68,0.51,0.50,0.31,0.21,0.20
TotalWorkingYears,0.68,1.00,0.78,0.77,0.63,0.46,0.46
JobLevel,0.51,0.78,1.00,0.95,0.53,0.39,0.38
MonthlyIncome,0.50,0.77,0.95,1.00,0.51,0.36,0.34
YearsAtCompany,0.31,0.63,0.53,0.51,1.00,0.76,0.77
YearsInCurrentRole,0.21,0.46,0.39,0.36,0.76,1.00,0.71
YearsWithCurrManager,0.20,0.46,0.38,0.34,0.77,0.71,1.00


**Se forman dos grupos:**

- **Nivel en la carrera:** `JobLevel` y `MonthlyIncome` (0,95) son prácticamente la misma variable, y `TotalWorkingYears` se mueve con ellas (0,77–0,78).
- **Tiempo en la empresa:** `YearsAtCompany`, `YearsInCurrentRole` y `YearsWithCurrManager` (0,71–0,77).
- `Age` se relaciona con los años de experiencia (0,68), pero menos con el resto.

Incluir variables tan relacionadas en un modelo (**multicolinealidad**) hace que sus resultados sean inestables y difíciles de interpretar. Por eso se elige una por grupo:

- **`MonthlyIncome`**: el sueldo es una palanca que la empresa puede ajustar.
- **`YearsAtCompany`**: la más simple de interpretar y comparable con el análisis en SQL.
- **`Age`**: aporta información propia.

## 6. Modelo: regresión logística

El modelo estima el efecto de cada variable **manteniendo las demás constantes**, algo que las tablas cruzadas no permiten con más de dos factores.

Las variables se estandarizan (`StandardScaler`) para que los coeficientes sean comparables entre sí. Coeficiente positivo = aumenta el riesgo de renuncia; negativo = lo reduce. Mientras más lejos de 0, mayor el efecto.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

variables = [
    "horas_extra",
    "MonthlyIncome",
    "YearsAtCompany",
    "Age",
    "StockOptionLevel",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "WorkLifeBalance",
    "DistanceFromHome",
    "NumCompaniesWorked",
]

X = df[variables]
y = df["renuncio"]

X_esc = StandardScaler().fit_transform(X)

modelo = LogisticRegression()
modelo.fit(X_esc, y)

coeficientes = pd.Series(modelo.coef_[0], index=variables).sort_values()
coeficientes.round(2)

MonthlyIncome             -0.47
StockOptionLevel          -0.45
Age                       -0.41
JobInvolvement            -0.39
EnvironmentSatisfaction   -0.38
JobSatisfaction           -0.34
WorkLifeBalance           -0.17
YearsAtCompany            -0.11
DistanceFromHome           0.27
NumCompaniesWorked         0.33
horas_extra                0.73
dtype: float64

**Interpretación:**

1. **Las horas extra son el factor principal (0,73)**, con un efecto mucho mayor que cualquier otra variable. El hallazgo se confirma en SQL, en la correlación y en el modelo.
2. **Un segundo grupo de factores con peso similar (-0,34 a -0,47):**
   - Económicos: **sueldo** y **acciones de la empresa**.
   - **Edad**: los empleados jóvenes renuncian más, incluso con igual sueldo y antigüedad.
   - Experiencia laboral: **involucramiento** y **satisfacción** con el ambiente y con el trabajo. En la correlación parecían débiles, pero cada uno tiene efecto propio.
3. **La antigüedad casi no pesa (-0,11).** En SQL, el primer año mostraba una tasa de renuncia de 34,9%. El modelo indica que, comparando empleados de igual edad y sueldo, los años en la empresa influyen poco: **los empleados nuevos renuncian más principalmente porque son más jóvenes y ganan menos**.
4. **Factores nuevos:** haber trabajado en **muchas empresas** (+0,33) y la **distancia de la casa** (+0,27) aumentan el riesgo. La distancia casi no aparecía en la correlación, pero pesa al comparar empleados similares.

*Limitación:* el modelo supone efectos lineales (cada año de antigüedad pesa lo mismo). Si el riesgo se concentra solo en el primer año, el modelo podría subestimarlo.

## 7. Evaluación del modelo

Se entrena con el 75% de los empleados y se evalúa con el 25% restante (368), que el modelo no vio.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo_eval = make_pipeline(StandardScaler(), LogisticRegression())
modelo_eval.fit(X_train, y_train)

pred = modelo_eval.predict(X_test)
prob = modelo_eval.predict_proba(X_test)[:, 1]

print("Exactitud:", round(accuracy_score(y_test, pred), 3))
print("AUC:", round(roc_auc_score(y_test, prob), 3))
confusion_matrix(y_test, pred)

Exactitud: 0.856
AUC: 0.782


array([[302,   7],
       [ 46,  13]])

- **AUC de 0,78:** el modelo ordena bien a los empleados según su riesgo (0,5 = azar; sobre 0,75 = bueno para datos de personas).
- **La exactitud (85,6%) engaña:** un modelo que dijera que nadie renuncia acertaría el 84%, porque la mayoría se queda.
- La **matriz de confusión** muestra el problema:

| | Predice "se queda" | Predice "renuncia" |
|---|---:|---:|
| **Se quedó** | 302 | 7 (falsa alarma) |
| **Renunció** | 46 (no detectado) | 13 |

De 59 renuncias, el modelo solo detecta 13 (22%). Como renunciar es poco común, el modelo casi nunca supera el umbral de 50% para predecir una renuncia.

## 8. Ajuste por desbalance

Con `class_weight="balanced"`, el modelo da más peso a las renuncias durante el aprendizaje. Se agregan dos métricas:

- **Detección (recall):** de quienes renunciaron, qué porcentaje detecta el modelo.
- **Precisión:** de quienes el modelo marca como riesgo, qué porcentaje realmente renuncia.

In [18]:
from sklearn.metrics import recall_score, precision_score

modelo_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced")
)
modelo_bal.fit(X_train, y_train)

pred_bal = modelo_bal.predict(X_test)
prob_bal = modelo_bal.predict_proba(X_test)[:, 1]

print("Exactitud:", round(accuracy_score(y_test, pred_bal), 3))
print("AUC:", round(roc_auc_score(y_test, prob_bal), 3))
print("Detección (recall):", round(recall_score(y_test, pred_bal), 3))
print("Precisión:", round(precision_score(y_test, pred_bal), 3))
confusion_matrix(y_test, pred_bal)

Exactitud: 0.736
AUC: 0.79
Detección (recall): 0.644
Precisión: 0.333


array([[233,  76],
       [ 21,  38]])

| Métrica | Modelo base | Modelo ajustado |
|---|---:|---:|
| Detección | 22% (13 de 59) | **64% (38 de 59)** |
| Precisión | 65% | 33% |
| AUC | 0,78 | 0,79 |
| Exactitud | 85,6% | 73,6% |

- El modelo ajustado **detecta casi el triple de renuncias**.
- Su precisión de 33% es **el doble que el azar** (16%).
- El AUC se mantiene: el orden de riesgo es el mismo; solo cambia desde qué nivel se considera "renuncia".

**Lectura para el negocio:** si el área de personas conversa con los empleados marcados como riesgo (cerca de 3 de cada 10), puede llegar a tiempo a casi 2 de cada 3 personas que se iban a ir. A cambio, 2 de cada 3 conversaciones serán con personas que no pensaban irse. Dado que reemplazar a un empleado suele costar bastante más que una conversación de retención, es un costo razonable, aunque la decisión final corresponde al negocio.

## 9. Puntaje de riesgo por empleado

Se entrena el modelo final con todos los empleados y se calcula un **puntaje de riesgo** para cada uno, clasificado en Bajo (< 0,3), Medio (0,3–0,6) y Alto (> 0,6). El resultado se guarda en MySQL en la tabla `riesgo_renuncia`, para usarlo en Power BI.

El conteo final considera solo a los **empleados activos**: en la práctica, el modelo se entrena con la historia y se aplica a quienes siguen en la empresa.

In [19]:
# 1. Modelo final con todos los datos
modelo_final = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced")
)
modelo_final.fit(X, y)

# 2. Puntaje de riesgo de cada empleado
df["puntaje_riesgo"] = modelo_final.predict_proba(X)[:, 1].round(3)

# 3. Categoría de riesgo
df["nivel_riesgo"] = pd.cut(
    df["puntaje_riesgo"],
    bins=[0, 0.3, 0.6, 1],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True,
)

# 4. Tabla de resultados
riesgo = df[["EmployeeNumber", "renuncio", "puntaje_riesgo", "nivel_riesgo"]]

# 5. Guardar en MySQL
riesgo.to_sql("riesgo_renuncia", engine, if_exists="replace", index=False)

# 6. Revisión: cuántos empleados activos hay en cada nivel
riesgo[riesgo["renuncio"] == 0]["nivel_riesgo"].value_counts()

nivel_riesgo
Bajo     546
Medio    494
Alto     193
Name: count, dtype: int64

- De 1.233 empleados activos, **193 (16%) quedan en riesgo alto**: es la lista prioritaria para acciones de retención.
- Entre los 10 puntajes más altos, 9 corresponden a empleados que efectivamente renunciaron. El restante (`EmployeeNumber` 1605, puntaje 0,946) sigue en la empresa y sería el primer caso a revisar.
- Este resultado es optimista porque el modelo final se entrenó con esos mismos empleados. La medida real de su rendimiento es la de la sección 8.
- Con `class_weight="balanced"`, los valores se inflan a propósito. Por eso se usan como **puntaje para ordenar y priorizar**, no como probabilidad literal de renuncia.

## Conclusiones

1. **Las horas extra son el factor más importante** en todos los análisis: tablas cruzadas, correlación y modelo.
2. **Sueldo, acciones, edad, involucramiento y satisfacción** tienen efectos propios de magnitud similar.
3. **La alta renuncia del primer año se explica principalmente por la edad y el sueldo** de los empleados nuevos, más que por la antigüedad en sí.
4. **El modelo ajustado detecta 2 de cada 3 renuncias** con el doble de certeza que el azar, y permite priorizar a los empleados activos según su riesgo.